In [2]:
import json

datastore = []
with open('/home/joao/tensorflow-journey/nlp/sarcasm.json', 'r') as f:
    for line in f:
        datastore.append(json.loads(line))


separe setences,
labels and urls

In [3]:
senteces = []
labels = []
urls = []

for item in datastore:
    senteces.append(item['headline'])
    labels.append(item['is_sarcastic'])
    urls.append(item['article_link'])

print(senteces[0])
print(labels[0])
print(urls[0])

former versace store clerk sues over secret 'black code' for minority shoppers
0
https://www.huffingtonpost.com/entry/versace-black-code_us_5861fbefe4b0de3a08f600d5


In [4]:
import tensorflow as tf

vectorize_layer = tf.keras.layers.TextVectorization()

vectorize_layer.adapt(senteces)

vocab = vectorize_layer.get_vocabulary()

post_padded_sequences = vectorize_layer(senteces)

print(f' padded: {post_padded_sequences[0]}')

2025-09-22 20:59:44.734570: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


 padded: [  295 15335   801  3788  2264    48   362    93  2225     6  2578  8719
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0]


In [5]:
training_size = 20000

training_sentences = senteces[0:training_size]
testing_sentences = senteces[training_size:]
training_labels = labels[0:training_size]
testing_labels = labels[training_size:]

vectorize_layer = tf.keras.layers.TextVectorization(
    max_tokens=10000,
    output_sequence_length=16
)

vectorize_layer.adapt(training_sentences)

train_sequences = vectorize_layer(training_sentences)
test_sequences = vectorize_layer(testing_sentences)

print(f'Padded train sequence: {train_sequences[0]}')
print(f'Padded test sequence: {test_sequences[0]}')

train_ds_vectorized = tf.data.Dataset.from_tensor_slices((train_sequences, training_labels))
test_ds_vectorized = tf.data.Dataset.from_tensor_slices((test_sequences, testing_labels))

train_ds_final = train_ds_vectorized.shuffle(10000).batch(32).prefetch(tf.data.AUTOTUNE)

test_ds_final = test_ds_vectorized.batch(32).prefetch(tf.data.AUTOTUNE)

model = tf.keras.Sequential([
    tf.keras.layers.Embedding(input_dim=10000, output_dim=16, input_length=16),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

model.summary()

history = model.fit(train_ds_final, epochs=10, validation_data=test_ds_final)

Padded train sequence: [ 319    1  943 4079 2366   47  366   94 2026    6 2653 9470    0    0
    0    0]
Padded test sequence: [   1 1126 6729 5062   30 8985 2158    5  675   88    0    0    0    0
    0    0]


/home/joao/.local/lib/python3.10/site-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.7468 - loss: 0.5129 - val_accuracy: 0.8346 - val_loss: 0.3817
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8777 - loss: 0.2988 - val_accuracy: 0.8520 - val_loss: 0.3455
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9130 - loss: 0.2246 - val_accuracy: 0.8527 - val_loss: 0.3560
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9325 - loss: 0.1801 - val_accuracy: 0.8463 - val_loss: 0.3863
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9462 - loss: 0.1505 - val_accuracy: 0.8435 - val_loss: 0.4150
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9567 - loss: 0.1268 - val_accuracy: 0.8408 - val_loss: 0.4545
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9628 - loss: 0.1102 - val_accuracy: 0.8328 - val_loss: 0.4962
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9680 - loss: 0.0948 - val_accuracy: 0.